# Ajuster la voix pidgin sur Common Voice camerounais (Kaggle)

Même entraînement que `entrainement_tts_colab.ipynb`, adapté à Kaggle. La voix de base (MMS-TTS pcm) parle **pidgin nigérian** ; on l'ajuste sur 12 voix camerounaises de Common Voice. La voix finale est un **mélange** des voix (aucun contributeur n'est reproduit).

**Une seule fois :**
1. Compte Kaggle **vérifié par téléphone** (sinon pas de GPU ni d'Internet).
2. **Datasets → New Dataset** : envoyer `KAMA_tts_wes_drive.zip`, titre `kama-tts-wes`, visibilité **Private** (données Common Voice : jamais public). Kaggle peut décompresser le zip : le carnet accepte les deux formes.
3. Dans ce carnet : **Add Input** → ton dataset `kama-tts-wes`.
4. **Settings** (panneau de droite) : **Accelerator = GPU T4 x2** (pas P100), **Internet = On**.

**Lancer :**
1. **Essai** : `SMOKE = True`, puis *Run All* en session interactive (quelques minutes). Noter le **temps par pas** (`s/it`) affiché dans la barre `Steps` de la section 5.
2. **Vrai entraînement** : `SMOKE = False`, régler `EPOCHS` avec le tableau ci-dessous, puis **Save Version → Save & Run All (Commit)**. Le carnet tourne alors sur les serveurs de Kaggle, navigateur fermé.
3. À la fin : onglet **Output** de la version → télécharger `mms-tts-wes-v1.zip` et `ecoute.zip`.

**Choisir `EPOCHS`** (une session Kaggle est coupée à 12 h, et rien n'est gardé avant la fin ; on vise 9 h au plus) : `EPOCHS = 9 × 3600 ÷ (284 × temps par pas)`, au plus 30.

| Temps par pas | EPOCHS |
|---|---|
| 3 s ou moins | 30 |
| 4 s | 28 |
| 5 s | 22 |
| 6 s | 19 |
| 8 s | 14 |
| 10 s | 11 |

**Reprendre un entraînement interrompu ou le prolonger (sans repartir de zéro)**

L'entraînement enregistre un **point de reprise** toutes les 250 étapes dans `/kaggle/working/tts_wes_sortie/checkpoint-N` (modèle, discriminateur, optimiseurs, planificateur : tout l'état). Les deux derniers sont gardés (environ 1 Go chacun).
1. Session 1 : `REPRISE = False`, `EPOCHS` = nombre de passes **total** visé pour cette session (ex. 10), **Save & Run All**. Viser moins de 9 h.
2. Session suivante : **Add Input → Your Work → ce carnet → la version précédente** (ses fichiers de sortie apparaissent dans `/kaggle/input/`). Mettre `REPRISE = True` et `EPOCHS` = nouveau **total cumulé** (ex. 20 : 10 de plus), puis **Save & Run All**. Le carnet recopie les points de reprise et l'entraînement repart du dernier, avec le même taux d'apprentissage et les mêmes optimiseurs.
3. Seule la passe en cours au moment de la coupure est refaite depuis son début.

Une session coupée par la limite de 12 h peut ne pas garder ses fichiers : terminer chaque session avant (réduire `EPOCHS`).

Licence : le modèle de base est CC-BY-NC 4.0 (non commercial) ; le modèle ajusté en hérite.

## 1. Réglages

In [ ]:
import os, sys
print('Python', sys.version.split()[0])

SMOKE = True      # True : essai rapide (6 enregistrements par locuteur). False : vrai entraînement.
EPOCHS = 11       # passes TOTALES visées à la fin de cette session (284 pas par passe) : voir le tableau ci-dessus
REPRISE = False   # True : repartir des points de reprise d'une version précédente de ce carnet (Add Input)

TMP = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'   # disque de travail (non gardé)
SORTIE_FINALE = '/kaggle/working'                                     # gardé dans l'onglet Output
LOCAL = f'{TMP}/KAMA/tts_wes'
SMOKE_DATA = f'{TMP}/smoke_data'
REPO = f'{TMP}/finetune-hf-vits'
BASE = f'{TMP}/mms-tts-pcm-train'     # modèle de base converti pour l'entraînement
SORTIE = f'{SORTIE_FINALE}/tts_wes_sortie'   # sortie de l'entraînement et points de reprise : gardés dans Output
FINAL = f'{TMP}/mms-tts-wes-v1'
ECOUTE = f'{TMP}/ecoute'
print('mode', 'ESSAI (SMOKE)' if SMOKE else f'ENTRAINEMENT COMPLET, {EPOCHS} passes au total', '| reprise' if REPRISE else '', '| travail :', TMP)

## 2. Données
Cherche le zip dans les données ajoutées au carnet (`/kaggle/input`), ou sa version déjà décompressée par Kaggle, et le copie sur le disque de travail.

In [ ]:
import glob, shutil, zipfile
shutil.rmtree(f'{TMP}/KAMA', ignore_errors=True)
zips = glob.glob('/kaggle/input/**/KAMA_tts_wes_drive.zip', recursive=True)
if zips:
    print('zip :', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        assert z.testzip() is None, 'zip abîmé : renvoie-le dans le dataset'
        z.extractall(TMP)
else:
    deja = glob.glob('/kaggle/input/**/tts_wes/data/train/metadata.csv', recursive=True)
    assert deja, "Données introuvables : ajoute le dataset kama-tts-wes au carnet (Add Input)"
    src = deja[0].rsplit('/data/train/metadata.csv', 1)[0]
    print('dossier déjà décompressé :', src)
    shutil.copytree(src, LOCAL)
for f in ('data/train/metadata.csv', 'data/test/metadata.csv', 'data/ecoute_phrases.json'):
    assert os.path.exists(f'{LOCAL}/{f}'), f'manquant : {f}'

# La colonne du locuteur s'appelle « locuteur », pas « speaker_id » : le script crée lui-même une colonne speaker_id
# (numéros 0..N-1) ; si elle existe déjà dans les données, elle reste masquée et l'entraînement plante (KeyError 'speaker_id').
import pandas as pd
def lire(split):
    p = f'{LOCAL}/data/{split}/metadata.csv'
    df = pd.read_csv(p, dtype=str, keep_default_na=False).rename(columns={'speaker_id': 'locuteur'})
    df.to_csv(p, index=False)
    return df
train, test = lire('train'), lire('test')
print('Données prêtes :', len(train), "enregistrements d'entraînement,", len(test), 'de test,', train.locuteur.nunique(), 'locuteurs')

## 2 bis. Jeu de données de l'essai (SMOKE seulement)
Les fichiers audio sont lus dans l'ordre de leur nom, qui commence par le code du locuteur : l'essai prend donc 6 enregistrements **par locuteur**.

In [ ]:
shutil.rmtree(SMOKE_DATA, ignore_errors=True)
if SMOKE:
    for split, df in (('train', train), ('test', test)):
        src = f'{LOCAL}/data/{split}'
        part = df.sample(frac=1, random_state=0).groupby('locuteur').head(6) if split == 'train' else df.head(8)
        os.makedirs(f'{SMOKE_DATA}/{split}/audio', exist_ok=True)
        for f in part.file_name:
            shutil.copy(f'{src}/{f}', f'{SMOKE_DATA}/{split}/{f}')
        part.to_csv(f'{SMOKE_DATA}/{split}/metadata.csv', index=False)
        print(split, len(part), 'enregistrements,', part.locuteur.nunique(), 'locuteurs')
else:
    print('Vrai entraînement : toutes les données, pas de jeu réduit.')

## 3. Installation
Outil d'ajustement de Hugging Face pour MMS-TTS (`ylacombe/finetune-hf-vits`). Versions fixées (transformers 4.x : la v5 a supprimé des réglages utilisés ici). Nécessite **Internet = On**.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets[audio]==2.21.0" "accelerate==0.34.2" soundfile librosa cython tensorboard
!rm -rf {REPO} && git clone -q https://github.com/ylacombe/finetune-hf-vits {REPO}
# matplotlib >= 3.10 n'a plus tostring_rgb (graphiques de validation) : on lit le tampon RGBA à la place
!sed -i 's/np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)/np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()/g' {REPO}/utils/plot.py
%cd {REPO}/monotonic_align
!mkdir -p monotonic_align && python setup.py build_ext --inplace
%cd {REPO}
!nvidia-smi -L

## 4. Modèle de base pour l'entraînement
Convertit le discriminateur de MMS-TTS pcm (téléchargé depuis `facebook/mms-tts`, environ 560 Mo) et l'assemble avec la voix de base.

In [ ]:
%cd {REPO}
!python convert_original_discriminator_checkpoint.py --language_code pcm --pytorch_dump_folder_path {BASE}
print(os.listdir(BASE))

## 4 bis. Reprise (si `REPRISE = True`)
Recopie la sortie de la version précédente (points de reprise compris) depuis `/kaggle/input/` vers `/kaggle/working/tts_wes_sortie`.

In [ ]:
import re as _re
if REPRISE:
    assert not SMOKE, 'pas de reprise en mode essai'
    trouves = sorted(glob.glob('/kaggle/input/**/tts_wes_sortie/checkpoint-*', recursive=True),
                     key=lambda c: int(_re.search(r'checkpoint-(\d+)', c).group(1)))
    assert trouves, "Aucun point de reprise : ajoute la version précédente de ce carnet (Add Input → Your Work)"
    precedent = os.path.dirname(trouves[-1])
    shutil.rmtree(SORTIE, ignore_errors=True)
    shutil.copytree(precedent, SORTIE)
    print('reprise depuis', trouves[-1], '| points disponibles :', [os.path.basename(c) for c in trouves])
else:
    shutil.rmtree(SORTIE, ignore_errors=True)
    print('nouvel entraînement')

## 5. Entraînement
Dans la sortie, cherche **« Resize speaker embeddings from 1 to 12 »**. En mode `SMOKE`, note le **temps par pas** (`s/it`) dans la barre `Steps`.
Un seul GPU est utilisé (`CUDA_VISIBLE_DEVICES=0`) : le second T4 n'accélère pas cet outil.

In [ ]:
import json
DONNEES = SMOKE_DATA if SMOKE else f'{LOCAL}/data'
cfg = {
    'project_name': 'kama_tts_wes', 'push_to_hub': False, 'overwrite_output_dir': True, 'output_dir': SORTIE,
    'dataset_name': DONNEES, 'audio_column_name': 'audio', 'text_column_name': 'text',
    'train_split_name': 'train', 'eval_split_name': 'test',
    'speaker_id_column_name': 'locuteur', 'override_speaker_embeddings': True,
    'max_duration_in_seconds': 12, 'min_duration_in_seconds': 1.5, 'max_tokens_length': 450,
    'model_name_or_path': BASE, 'preprocessing_num_workers': 2,
    'do_train': True, 'num_train_epochs': 1 if SMOKE else EPOCHS, 'gradient_accumulation_steps': 1, 'gradient_checkpointing': False,
    'per_device_train_batch_size': 16, 'learning_rate': 2e-5, 'adam_beta1': 0.8, 'adam_beta2': 0.99, 'warmup_ratio': 0.01, 'group_by_length': False,
    'do_eval': True, 'eval_steps': 2 if SMOKE else 284, 'per_device_eval_batch_size': 8, 'max_eval_samples': 8 if SMOKE else 40,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3, 'weight_fmaps': 1, 'weight_gen': 1, 'weight_kl': 1.5, 'weight_duration': 1, 'weight_mel': 35,
    'fp16': True, 'seed': 456, 'report_to': 'tensorboard',
    # Points de reprise : tout l'état (modèles, optimiseurs, planificateurs) toutes les 250 étapes ; les 2 derniers gardés.
    'save_steps': 1_000_000 if SMOKE else 250, 'save_total_limit': 2,
}
if REPRISE:
    cfg['resume_from_checkpoint'] = 'latest'   # le dernier checkpoint-N de SORTIE
json.dump(cfg, open(f'{TMP}/config_tts.json', 'w'), indent=1)
os.environ['WANDB_DISABLED'] = 'true'

n_loc = pd.read_csv(f'{DONNEES}/train/metadata.csv', dtype=str).locuteur.nunique()
assert n_loc > 1, f'{n_loc} locuteur dans {DONNEES} : relance les cellules 2 et 2 bis'
print('mode', 'ESSAI (SMOKE)' if SMOKE else f'ENTRAINEMENT COMPLET, {EPOCHS} passes')
print('données :', DONNEES, '|', n_loc, 'locuteurs')

In [ ]:
%cd {REPO}
!CUDA_VISIBLE_DEVICES=0 accelerate launch --num_processes 1 run_vits_finetuning.py {TMP}/config_tts.json

## 6. Voix « mélange »
On ajoute une voix égale à la **moyenne** des 12 : c'est celle que le serveur utilisera, pour qu'aucun contributeur ne soit reproduit. Le numéro à mettre dans `routes.json` est affiché à la fin.

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer
m = VitsModel.from_pretrained(SORTIE)
tok = AutoTokenizer.from_pretrained(SORTIE)
assert m.config.num_speakers > 1, 'le modèle entraîné n a qu une voix : vérifie la section 5'
w = m.embed_speaker.weight.data.clone()
n = w.shape[0]
m.embed_speaker = torch.nn.Embedding(n + 1, w.shape[1])
m.embed_speaker.weight.data = torch.cat([w, w.mean(dim=0, keepdim=True)])
m.config.num_speakers = n + 1
MIX_ID = n
shutil.rmtree(FINAL, ignore_errors=True)
m.save_pretrained(FINAL)
tok.save_pretrained(FINAL)
print(f'{n} voix + la voix mélange. speaker_id du mélange = {MIX_ID}  <- à mettre dans routes.json (options.speaker_id)')

## 7. Échantillons à écouter
Pour chaque phrase : `_base` (voix nigériane d'origine) et `_ajuste` (voix ajustée, mélange).

In [ ]:
import re, scipy.io.wavfile as wav
phrases = json.load(open(f'{LOCAL}/data/ecoute_phrases.json', encoding='utf-8'))
vocab = set(json.load(open(f'{FINAL}/vocab.json')))
def propre(t):
    return re.sub(r'\s+', ' ', ''.join(c if c in vocab else ' ' for c in t.lower())).strip()
base = VitsModel.from_pretrained('facebook/mms-tts-pcm').eval()
tok0 = AutoTokenizer.from_pretrained('facebook/mms-tts-pcm')
m.eval()
shutil.rmtree(ECOUTE, ignore_errors=True)
os.makedirs(ECOUTE)
for i, t in enumerate(phrases):
    texte = propre(t)
    if not texte:
        continue
    with torch.inference_mode():
        torch.manual_seed(0)
        a = base(**tok0(texte, return_tensors='pt')).waveform[0].numpy()
        torch.manual_seed(0)
        b = m(**tok(texte, return_tensors='pt'), speaker_id=MIX_ID).waveform[0].numpy()
    wav.write(f'{ECOUTE}/{i:02d}_base.wav', 16000, a)
    wav.write(f'{ECOUTE}/{i:02d}_ajuste.wav', 16000, b)
    print(i, texte)
print(len(os.listdir(ECOUTE)), 'fichiers dans', ECOUTE)

## 8. Récupérer le modèle
Les zips sont écrits dans `/kaggle/working` : onglet **Output** de la version (ou panneau de droite en session interactive) → télécharger. Sur ton poste, le modèle va dans `KAMA/models/poids/tts/mms-tts-wes-v1` (hors Git).

In [ ]:
suffixe = '_essai' if SMOKE else ''
for nom, dossier in ((f'mms-tts-wes-v1{suffixe}', FINAL), (f'ecoute{suffixe}', ECOUTE)):
    chemin = shutil.make_archive(f'{SORTIE_FINALE}/{nom}', 'zip', root_dir=os.path.dirname(dossier), base_dir=os.path.basename(dossier))
    print(chemin, round(os.path.getsize(chemin) / 1e6), 'Mo')